In [1]:
import sqlite3
import pandas as pd

In [2]:
conn = sqlite3.connect('../sql/olist.db')

tables = {
    'customers': '../data/processed/customers_clean.csv',
    'orders': '../data/processed/orders_clean.csv',
    'order_items': '../data/processed/order_items_clean.csv',
    'order_payments': '../data/processed/order_payments_clean.csv',
    'order_reviews': '../data/processed/order_reviews_clean.csv',
    'products': '../data/processed/products_clean.csv',
    'sellers': '../data/processed/sellers_clean.csv',
    'locations': '../data/processed/locations_clean.csv',
    'product_category': '../data/processed/product_category_clean.csv'
}

for table_name, path in tables.items():
    df = pd.read_csv(path)
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f'{table_name}: {len(df)} rows loaded')

conn.close()

customers: 99441 rows loaded
orders: 99441 rows loaded
order_items: 112650 rows loaded
order_payments: 103886 rows loaded
order_reviews: 99224 rows loaded
products: 32949 rows loaded
sellers: 3095 rows loaded
locations: 19015 rows loaded
product_category: 71 rows loaded


In [3]:
conn=sqlite3.connect('../sql/olist.db')

In [4]:
query = """
SELECT COUNT(DISTINCT orders.order_id) AS total_orders,
ROUND(SUM(price),2) AS total_revenue,
ROUND(SUM(price)/COUNT(DISTINCT orders.order_id),2) AS avg_order_revenue
FROM orders
JOIN order_items
ON orders.order_id=order_items.order_id
WHERE order_status = 'delivered';
"""

result = pd.read_sql(query, conn)
result

,total_orders,total_revenue,avg_order_revenue
0,96478,13221498.11,137.04


In [5]:
query = """
SELECT product_category.product_category_name_english,
SUM (order_items.price) AS total_revenue
FROM orders
JOIN order_items
ON orders.order_id=order_items.order_id
JOIN products
ON products.product_id=order_items.product_id
JOIN product_category
ON products.product_category_name=product_category.product_category_name
WHERE order_status = 'delivered'
GROUP BY product_category.product_category_name
ORDER BY total_revenue DESC
LIMIT 10;
"""

result = pd.read_sql(query, conn)
result

,product_category_name_english,total_revenue
0,health_beauty,1233131.72
1,watches_gifts,1166176.98
2,bed_bath_table,1023434.76
3,sports_leisure,954852.55
4,computers_accessories,888724.61
5,furniture_decor,711927.69
6,housewares,615628.69
7,cool_stuff,610204.10
8,auto,578966.65
9,toys,471286.48


In [6]:
query = """
SELECT customers.customer_state AS state,
SUM(order_items.price) AS revenue_by_state
FROM customers
JOIN orders
ON orders.customer_id=customers.customer_id
JOIN order_items
ON order_items.order_id=orders.order_id
WHERE order_status = 'delivered'
GROUP BY customers.customer_state
ORDER BY revenue_by_state DESC
LIMIT 10;
"""
result = pd.read_sql(query, conn)
result

,state,revenue_by_state
0,SP,5067633.16
1,RJ,1759651.13
2,MG,1552481.83
3,RS,728897.47
4,PR,666063.51
5,SC,507012.13
6,BA,493584.14
7,DF,296498.41
8,GO,282836.70
9,ES,268643.45


In [10]:
query = """
SELECT ROUND(AVG(FLOOR(julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date))),2) AS avg_delivery_delay,
ROUND(AVG(CASE WHEN FLOOR(julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date)) > 0 THEN 1 ELSE 0 END)*100,2) AS late_rate_percent
FROM orders
WHERE order_status = 'delivered';
"""
result = pd.read_sql(query, conn)
result

,avg_delivery_delay,late_rate_percent
0,-11.88,6.77
